In [ ]:
"""
==============================================================
  이탈 예측 최종 통합 모델 (churn_model_final.py)
  실험 결과 기반 베스트 피처 + 베스트 파라미터 집약본
  Target: ROC-AUC 최대화
==============================================================
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')


# ──────────────────────────────────────────────
# 0. 데이터 로드 (경로 수정)
# ──────────────────────────────────────────────
BASE = '/Users/rim/Desktop/workspace/project_1'

train_cust   = pd.read_csv(f'{BASE}/train/train_customer_info.csv')
train_trans  = pd.read_csv(f'{BASE}/train/train_transaction_history.csv')
train_fin    = pd.read_csv(f'{BASE}/train/train_finance_profile.csv')
train_target = pd.read_csv(f'{BASE}/train/train_targets.csv')

test_cust    = pd.read_csv(f'{BASE}/test/test_customer_info.csv')
test_trans   = pd.read_csv(f'{BASE}/test/test_transaction_history.csv')
test_fin     = pd.read_csv(f'{BASE}/test/test_finance_profile.csv')

print("✅ 데이터 로드 완료")


# ──────────────────────────────────────────────
# 1. 피처 엔지니어링 함수
# ──────────────────────────────────────────────
def build_features(cust_df, trans_df, fin_df, target_df=None):
    """
    고객 기본 정보 + 금융 프로필 + 거래 이력을 결합해
    모델 입력용 피처 데이터프레임을 반환합니다.
    target_df가 None이면 테스트 모드로 동작합니다.
    """

    # ── 1-1. 기본 병합 (1:1) ──────────────────
    df = pd.merge(cust_df, fin_df, on='customer_id', how='left')
    if target_df is not None:
        df = pd.merge(df, target_df, on='customer_id', how='left')

    # ── 1-2. 날짜 처리 ────────────────────────
    df['join_date']   = pd.to_datetime(df['join_date'])
    trans_df = trans_df.copy()
    trans_df['trans_date'] = pd.to_datetime(trans_df['trans_date'])
    trans_df['month'] = trans_df['trans_date'].dt.month

    ref_date = trans_df['trans_date'].max()
    df['days_since_joined'] = (ref_date - df['join_date']).dt.days

    # ── 1-3. 범주형 인코딩 ────────────────────
    cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # ── 1-4. 기본 파생 변수 ───────────────────
    df['net_asset']          = df['total_deposit_balance'] - df['total_loan_balance']
    df['debt_to_deposit']    = df['total_loan_balance'] / (df['total_deposit_balance'] + 1)
    df['cash_service_ratio'] = df['card_cash_service_amt'] / (df['total_deposit_balance'] + 1)

    # 재정 위기 복합 지표 (표준화 합산)
    distress_vars = ['fin_overdue_days', 'card_cash_service_amt',
                     'card_loan_amt', 'total_loan_balance']
    distress_z = df[distress_vars].copy()
    for v in distress_vars:
        mu, sd = distress_z[v].mean(), distress_z[v].std() + 1e-9
        distress_z[v] = (distress_z[v] - mu) / sd
    df['fin_distress_v2'] = distress_z.sum(axis=1)

    # 고위험 자산 플래그
    df['is_high_risk_asset'] = (
        (df['fin_asset_trend_score'] <= -2) & (df['credit_score'] < 600)
    ).astype(int)

    # ── 1-5. 거래 이력 집계 ───────────────────
    # 전체 집계
    agg = trans_df.groupby('customer_id').agg(
        trans_count      = ('trans_id',       'count'),
        amt_sum          = ('trans_amount',    'sum'),
        amt_mean         = ('trans_amount',    'mean'),
        amt_max          = ('trans_amount',    'max'),
        amt_std          = ('trans_amount',    'std'),
        installment_ratio= ('is_installment',  'mean'),
        online_ratio     = ('biz_type',        lambda x: (x == 'Online').mean()),
        recency          = ('trans_date',
                            lambda x: (ref_date - x.max()).days),
    ).reset_index()
    agg['spending_per_trans'] = agg['amt_sum'] / (agg['trans_count'] + 1)
    agg['amt_std'] = agg['amt_std'].fillna(0)

    df = pd.merge(df, agg, on='customer_id', how='left')
    fill_cols = ['trans_count','amt_sum','amt_mean','amt_max',
                 'amt_std','installment_ratio','online_ratio',
                 'recency','spending_per_trans']
    df[fill_cols] = df[fill_cols].fillna(0)

    # ── 1-6. 월별 Wide 피처 (핵심 시계열) ────
    monthly_amt = trans_df.groupby(['customer_id','month'])['trans_amount'] \
        .sum().unstack(fill_value=0)
    monthly_amt.columns = [f'amt_m{int(c)}' for c in monthly_amt.columns]

    monthly_cnt = trans_df.groupby(['customer_id','month'])['trans_id'] \
        .count().unstack(fill_value=0)
    monthly_cnt.columns = [f'cnt_m{int(c)}' for c in monthly_cnt.columns]

    df = df.merge(monthly_amt.reset_index(), on='customer_id', how='left')
    df = df.merge(monthly_cnt.reset_index(), on='customer_id', how='left')

    # 월별 컬럼 결측치 0 처리
    month_cols = [c for c in df.columns if c.startswith('amt_m') or c.startswith('cnt_m')]
    df[month_cols] = df[month_cols].fillna(0)

    # 후반(10~12월) vs 전반(7~9월) 소비 감소율
    first_half_cols  = [c for c in df.columns if c in ['amt_m7','amt_m8','amt_m9']]
    second_half_cols = [c for c in df.columns if c in ['amt_m10','amt_m11','amt_m12']]
    if first_half_cols and second_half_cols:
        df['amt_first_half']    = df[first_half_cols].sum(axis=1)
        df['amt_second_half']   = df[second_half_cols].sum(axis=1)
        df['half_decline_ratio'] = df['amt_second_half'] / (df['amt_first_half'] + 1)

    # 12월 거래 없는 고객 (강력한 이탈 선행 신호)
    last_month_buyers = trans_df[trans_df['month'] == trans_df['month'].max()]['customer_id'].unique()
    df['no_last_month_trans'] = (~df['customer_id'].isin(last_month_buyers)).astype(int)

    # 최근 소비 감소 플래그
    if 'amt_m12' in df.columns and 'amt_m11' in df.columns:
        df['recent_spending_drop'] = (df['amt_m12'] < df['amt_m11']).astype(int)

    # spending trend ratio (최근 1달 vs 이전 5달 평균)
    max_month = trans_df['month'].max()
    recent_sum = trans_df[trans_df['month'] == max_month] \
        .groupby('customer_id')['trans_amount'].sum()
    older_mean = trans_df[trans_df['month'] < max_month] \
        .groupby('customer_id')['trans_amount'].mean()
    df['spending_trend_ratio'] = (
        df['customer_id'].map(recent_sum).fillna(0) /
        (df['customer_id'].map(older_mean).fillna(1) + 1)
    )

    # ── 1-7. 엔트로피 피처 ───────────────────
    def calc_entropy(series):
        counts = series.value_counts(normalize=True)
        return scipy_entropy(counts)

    cat_ent = trans_df.groupby('customer_id')['item_category'].agg(calc_entropy)
    df['category_entropy'] = df['customer_id'].map(cat_ent).fillna(0)

    # ── 1-8. 피어 그룹 내 상대 순위 ──────────
    df['credit_rank_in_income']  = df.groupby('income_group')['credit_score'] \
                                     .rank(pct=True)
    df['deposit_rank_in_region'] = df.groupby('region_code')['total_deposit_balance'] \
                                     .rank(pct=True)
    df['age_group'] = pd.cut(df['age'], bins=[0,30,40,50,100],
                              labels=[0,1,2,3]).astype(float)
    df['loan_rank_in_age']       = df.groupby('age_group')['card_loan_amt'] \
                                     .rank(pct=True)

    # ── 1-9. 타겟 인코딩 (train only, OOF 방식) ─
    if target_df is not None:
        def target_encode_oof(df_, col, target_col='target_churn', n_splits=5):
            encoded = pd.Series(index=df_.index, dtype=float)
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            for tr_idx, val_idx in kf.split(df_):
                mean_map = df_.iloc[tr_idx].groupby(col)[target_col].mean()
                encoded.iloc[val_idx] = df_.iloc[val_idx][col].map(mean_map)
            return encoded.fillna(df_[target_col].mean())

        for col in ['region_code', 'prefer_category', 'income_group']:
            if col in df.columns:
                df[f'{col}_churn_rate'] = target_encode_oof(df, col)

    return df


# ──────────────────────────────────────────────
# 2. 피처 목록 정의
# ──────────────────────────────────────────────
FEATURES = [
    # 금융 핵심
    'total_deposit_balance', 'card_loan_amt', 'credit_score',
    'fin_distress_v2', 'net_asset', 'fin_asset_trend_score',
    'num_active_cards', 'fin_overdue_days',
    'debt_to_deposit', 'cash_service_ratio',

    # 거래 집계
    'trans_count', 'amt_sum', 'amt_mean', 'amt_std',
    'spending_per_trans', 'installment_ratio', 'online_ratio', 'recency',

    # 시계열 (월별 wide)
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    'half_decline_ratio', 'no_last_month_trans',
    'recent_spending_drop', 'spending_trend_ratio',

    # 피어 그룹 순위
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',

    # 엔트로피
    'category_entropy',

    # 고객 속성
    'age', 'is_married', 'days_since_joined', 'is_high_risk_asset',
]


# ──────────────────────────────────────────────
# 3. 학습 데이터 구축
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (train)...")
train_df = build_features(train_cust, train_trans, train_fin, train_target)

# 타겟 인코딩 피처 추가
te_features = [f'{c}_churn_rate' for c in ['region_code','prefer_category','income_group']
               if f'{c}_churn_rate' in train_df.columns]

final_features = [f for f in FEATURES + te_features if f in train_df.columns]

X = train_df[final_features].copy()
y = train_df['target_churn']

print(f"✅ 학습 피처 수: {len(final_features)}개 | 샘플 수: {len(X):,}명")
print(f"   이탈율: {y.mean():.3%}")


# ──────────────────────────────────────────────
# 4. 모델 학습 (Optuna Trial 31 베스트 파라미터)
# ──────────────────────────────────────────────
BEST_PARAMS = {
    'objective':       'binary',
    'metric':          'auc',
    'verbosity':       -1,
    'boosting_type':   'gbdt',
    'random_state':    42,
    'n_estimators':    5000,
    # Optuna 최적값
    'learning_rate':   0.014369,
    'num_leaves':      31,
    'min_child_samples': 22,
    'feature_fraction':  0.7661,
    'bagging_fraction':  0.7553,
    'bagging_freq':      6,
    'reg_alpha':         0.0172,
    'reg_lambda':        0.0525,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models    = []
fold_aucs = []

print("\n🚀 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False),
                   lgb.log_evaluation(0)]
    )
    models.append(model)

    pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = pred
    auc = roc_auc_score(y_val, pred)
    fold_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC = {auc:.4f}")

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  CV AUC (OOF): {cv_auc:.4f}")
print(f"  Fold 평균:    {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"{'='*45}")

# 피처 중요도 Top 15
importances = pd.Series(
    np.mean([m.feature_importances_ for m in models], axis=0),
    index=final_features
).sort_values(ascending=False)
print("\n🔥 피처 중요도 Top 15:")
print(importances.head(15).to_string())


# ──────────────────────────────────────────────
# 5. 테스트 예측 및 제출 파일 생성
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (test)...")
test_df = build_features(test_cust, test_trans, test_fin, target_df=None)

# 타겟 인코딩 — train 전체 통계로 매핑 (test는 OOF 불필요)
for col in ['region_code', 'prefer_category', 'income_group']:
    col_te = f'{col}_churn_rate'
    if col_te in final_features:
        train_mean_map = train_df.groupby(col)['target_churn'].mean()
        test_df[col_te] = test_df[col].map(train_mean_map).fillna(y.mean())

# 없는 컬럼 0으로 패딩
for f in final_features:
    if f not in test_df.columns:
        test_df[f] = 0

X_test = test_df[final_features].copy()

# 5개 fold 모델 평균으로 최종 예측
test_preds = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)

# 제출 파일 저장
submission = pd.DataFrame({
    'customer_id':  test_df['customer_id'],
    'target_churn': test_preds,
    'target_ltv':   0.0   # LTV는 별도 모델 필요 — 일단 0으로 플레이스홀더
})
submission.to_csv(f'{BASE}/submission_churn.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ 제출 파일 저장 완료: {BASE}/submission_churn.csv")
print(f"   예측값 범위: {test_preds.min():.4f} ~ {test_preds.max():.4f}")
print(f"   예측 이탈율: {test_preds.mean():.3%}")

In [1]:
"""
==============================================================
  이탈 예측 최종 통합 모델 (churn_model_final.py)
  실험 결과 기반 베스트 피처 + 베스트 파라미터 집약본
  Target: ROC-AUC 최대화
==============================================================
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')


# ──────────────────────────────────────────────
# 0. 데이터 로드 (경로 수정)
# ──────────────────────────────────────────────
BASE = '/Users/rim/Desktop/workspace/project_1'

train_cust   = pd.read_csv(f'{BASE}/train/train_customer_info.csv')
train_trans  = pd.read_csv(f'{BASE}/train/train_transaction_history.csv')
train_fin    = pd.read_csv(f'{BASE}/train/train_finance_profile.csv')
train_target = pd.read_csv(f'{BASE}/train/train_targets.csv')

test_cust    = pd.read_csv(f'{BASE}/test/test_customer_info.csv')
test_trans   = pd.read_csv(f'{BASE}/test/test_transaction_history.csv')
test_fin     = pd.read_csv(f'{BASE}/test/test_finance_profile.csv')

print("✅ 데이터 로드 완료")


# ──────────────────────────────────────────────
# 1. 피처 엔지니어링 함수
# ──────────────────────────────────────────────
def build_features(cust_df, trans_df, fin_df, target_df=None):
    """
    고객 기본 정보 + 금융 프로필 + 거래 이력을 결합해
    모델 입력용 피처 데이터프레임을 반환합니다.
    target_df가 None이면 테스트 모드로 동작합니다.
    """

    # ── 1-1. 기본 병합 (1:1) ──────────────────
    df = pd.merge(cust_df, fin_df, on='customer_id', how='left')
    if target_df is not None:
        df = pd.merge(df, target_df, on='customer_id', how='left')

    # ── 1-2. 날짜 처리 ────────────────────────
    df['join_date']   = pd.to_datetime(df['join_date'])
    trans_df = trans_df.copy()
    trans_df['trans_date'] = pd.to_datetime(trans_df['trans_date'])
    trans_df['month'] = trans_df['trans_date'].dt.month

    ref_date = trans_df['trans_date'].max()
    df['days_since_joined'] = (ref_date - df['join_date']).dt.days

    # ── 1-3. 범주형 인코딩 ────────────────────
    cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # ── 1-4. 기본 파생 변수 ───────────────────
    df['net_asset']          = df['total_deposit_balance'] - df['total_loan_balance']
    df['debt_to_deposit']    = df['total_loan_balance'] / (df['total_deposit_balance'] + 1)
    df['cash_service_ratio'] = df['card_cash_service_amt'] / (df['total_deposit_balance'] + 1)

    # 재정 위기 복합 지표 (표준화 합산)
    distress_vars = ['fin_overdue_days', 'card_cash_service_amt',
                     'card_loan_amt', 'total_loan_balance']
    distress_z = df[distress_vars].copy()
    for v in distress_vars:
        mu, sd = distress_z[v].mean(), distress_z[v].std() + 1e-9
        distress_z[v] = (distress_z[v] - mu) / sd
    df['fin_distress_v2'] = distress_z.sum(axis=1)

    # 고위험 자산 플래그
    df['is_high_risk_asset'] = (
        (df['fin_asset_trend_score'] <= -2) & (df['credit_score'] < 600)
    ).astype(int)

    # ── 1-5. 거래 이력 집계 ───────────────────
    # 전체 집계
    agg = trans_df.groupby('customer_id').agg(
        trans_count      = ('trans_id',       'count'),
        amt_sum          = ('trans_amount',    'sum'),
        amt_mean         = ('trans_amount',    'mean'),
        amt_max          = ('trans_amount',    'max'),
        amt_std          = ('trans_amount',    'std'),
        installment_ratio= ('is_installment',  'mean'),
        online_ratio     = ('biz_type',        lambda x: (x == 'Online').mean()),
        recency          = ('trans_date',
                            lambda x: (ref_date - x.max()).days),
    ).reset_index()
    agg['spending_per_trans'] = agg['amt_sum'] / (agg['trans_count'] + 1)
    agg['amt_std'] = agg['amt_std'].fillna(0)

    df = pd.merge(df, agg, on='customer_id', how='left')
    fill_cols = ['trans_count','amt_sum','amt_mean','amt_max',
                 'amt_std','installment_ratio','online_ratio',
                 'recency','spending_per_trans']
    df[fill_cols] = df[fill_cols].fillna(0)

    # ── 1-6. 월별 Wide 피처 (핵심 시계열) ────
    monthly_amt = trans_df.groupby(['customer_id','month'])['trans_amount'] \
        .sum().unstack(fill_value=0)
    monthly_amt.columns = [f'amt_m{int(c)}' for c in monthly_amt.columns]

    monthly_cnt = trans_df.groupby(['customer_id','month'])['trans_id'] \
        .count().unstack(fill_value=0)
    monthly_cnt.columns = [f'cnt_m{int(c)}' for c in monthly_cnt.columns]

    df = df.merge(monthly_amt.reset_index(), on='customer_id', how='left')
    df = df.merge(monthly_cnt.reset_index(), on='customer_id', how='left')

    # 월별 컬럼 결측치 0 처리
    month_cols = [c for c in df.columns if c.startswith('amt_m') or c.startswith('cnt_m')]
    df[month_cols] = df[month_cols].fillna(0)

    # 후반(10~12월) vs 전반(7~9월) 소비 감소율
    first_half_cols  = [c for c in df.columns if c in ['amt_m7','amt_m8','amt_m9']]
    second_half_cols = [c for c in df.columns if c in ['amt_m10','amt_m11','amt_m12']]
    if first_half_cols and second_half_cols:
        df['amt_first_half']    = df[first_half_cols].sum(axis=1)
        df['amt_second_half']   = df[second_half_cols].sum(axis=1)
        df['half_decline_ratio'] = df['amt_second_half'] / (df['amt_first_half'] + 1)

    # 12월 거래 없는 고객 (강력한 이탈 선행 신호)
    last_month_buyers = trans_df[trans_df['month'] == trans_df['month'].max()]['customer_id'].unique()
    df['no_last_month_trans'] = (~df['customer_id'].isin(last_month_buyers)).astype(int)

    # 최근 소비 감소 플래그
    if 'amt_m12' in df.columns and 'amt_m11' in df.columns:
        df['recent_spending_drop'] = (df['amt_m12'] < df['amt_m11']).astype(int)

    # spending trend ratio (최근 1달 vs 이전 5달 평균)
    max_month = trans_df['month'].max()
    recent_sum = trans_df[trans_df['month'] == max_month] \
        .groupby('customer_id')['trans_amount'].sum()
    older_mean = trans_df[trans_df['month'] < max_month] \
        .groupby('customer_id')['trans_amount'].mean()
    df['spending_trend_ratio'] = (
        df['customer_id'].map(recent_sum).fillna(0) /
        (df['customer_id'].map(older_mean).fillna(1) + 1)
    )

    # ── 1-7. 엔트로피 피처 ───────────────────
    def calc_entropy(series):
        counts = series.value_counts(normalize=True)
        return scipy_entropy(counts)

    cat_ent = trans_df.groupby('customer_id')['item_category'].agg(calc_entropy)
    df['category_entropy'] = df['customer_id'].map(cat_ent).fillna(0)

    # ── 1-8. 피어 그룹 내 상대 순위 ──────────
    df['credit_rank_in_income']  = df.groupby('income_group')['credit_score'] \
                                     .rank(pct=True)
    df['deposit_rank_in_region'] = df.groupby('region_code')['total_deposit_balance'] \
                                     .rank(pct=True)
    df['age_group'] = pd.cut(df['age'], bins=[0,30,40,50,100],
                              labels=[0,1,2,3]).astype(float)
    df['loan_rank_in_age']       = df.groupby('age_group')['card_loan_amt'] \
                                     .rank(pct=True)

    # ── 1-9. 타겟 인코딩 (train only, OOF 방식) ─
    if target_df is not None:
        def target_encode_oof(df_, col, target_col='target_churn', n_splits=5):
            encoded = pd.Series(index=df_.index, dtype=float)
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            for tr_idx, val_idx in kf.split(df_):
                mean_map = df_.iloc[tr_idx].groupby(col)[target_col].mean()
                encoded.iloc[val_idx] = df_.iloc[val_idx][col].map(mean_map)
            return encoded.fillna(df_[target_col].mean())

        for col in ['region_code', 'prefer_category', 'income_group']:
            if col in df.columns:
                df[f'{col}_churn_rate'] = target_encode_oof(df, col)

    return df


# ──────────────────────────────────────────────
# 2. 피처 목록 정의
# ──────────────────────────────────────────────
FEATURES = [
    # 금융 핵심
    'total_deposit_balance', 'card_loan_amt', 'credit_score',
    'fin_distress_v2', 'net_asset', 'fin_asset_trend_score',
    'num_active_cards', 'fin_overdue_days',
    'debt_to_deposit', 'cash_service_ratio',

    # 거래 집계
    'trans_count', 'amt_sum', 'amt_mean', 'amt_std',
    'spending_per_trans', 'installment_ratio', 'online_ratio', 'recency',

    # 시계열 (월별 wide)
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    'half_decline_ratio', 'no_last_month_trans',
    'recent_spending_drop', 'spending_trend_ratio',

    # 피어 그룹 순위
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',

    # 엔트로피
    'category_entropy',

    # 고객 속성
    'age', 'is_married', 'days_since_joined', 'is_high_risk_asset',
]


# ──────────────────────────────────────────────
# 3. 학습 데이터 구축
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (train)...")
train_df = build_features(train_cust, train_trans, train_fin, train_target)

# 타겟 인코딩 피처 추가
te_features = [f'{c}_churn_rate' for c in ['region_code','prefer_category','income_group']
               if f'{c}_churn_rate' in train_df.columns]

final_features = [f for f in FEATURES + te_features if f in train_df.columns]

X = train_df[final_features].copy()
y = train_df['target_churn']

print(f"✅ 학습 피처 수: {len(final_features)}개 | 샘플 수: {len(X):,}명")
print(f"   이탈율: {y.mean():.3%}")


# ──────────────────────────────────────────────
# 4. 모델 학습 (Optuna Trial 31 베스트 파라미터)
# ──────────────────────────────────────────────
BEST_PARAMS = {
    'objective':       'binary',
    'metric':          'auc',
    'verbosity':       -1,
    'boosting_type':   'gbdt',
    'random_state':    42,
    'n_estimators':    5000,
    # Optuna 최적값
    'learning_rate':   0.014369,
    'num_leaves':      31,
    'min_child_samples': 22,
    'feature_fraction':  0.7661,
    'bagging_fraction':  0.7553,
    'bagging_freq':      6,
    'reg_alpha':         0.0172,
    'reg_lambda':        0.0525,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models    = []
fold_aucs = []

print("\n🚀 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False),
                   lgb.log_evaluation(0)]
    )
    models.append(model)

    pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = pred
    auc = roc_auc_score(y_val, pred)
    fold_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC = {auc:.4f}")

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  CV AUC (OOF): {cv_auc:.4f}")
print(f"  Fold 평균:    {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"{'='*45}")

# 피처 중요도 Top 15
importances = pd.Series(
    np.mean([m.feature_importances_ for m in models], axis=0),
    index=final_features
).sort_values(ascending=False)
print("\n🔥 피처 중요도 Top 15:")
print(importances.head(15).to_string())


# ──────────────────────────────────────────────
# 5. 테스트 예측 및 제출 파일 생성
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (test)...")
test_df = build_features(test_cust, test_trans, test_fin, target_df=None)

# 타겟 인코딩 — train 전체 통계로 매핑 (test는 OOF 불필요)
for col in ['region_code', 'prefer_category', 'income_group']:
    col_te = f'{col}_churn_rate'
    if col_te in final_features:
        train_mean_map = train_df.groupby(col)['target_churn'].mean()
        test_df[col_te] = test_df[col].map(train_mean_map).fillna(y.mean())

# 없는 컬럼 0으로 패딩
for f in final_features:
    if f not in test_df.columns:
        test_df[f] = 0

X_test = test_df[final_features].copy()

# 5개 fold 모델 평균으로 최종 예측
test_preds = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)

# 제출 파일 저장
# submission = pd.DataFrame({
    'customer_id':  test_df['customer_id'],
    'target_churn': test_preds,
    'target_ltv':   0.0   # LTV는 별도 모델 필요 — 일단 0으로 플레이스홀더
})
# submission.to_csv(f'{BASE}/submission_churn.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ 제출 파일 저장 완료: {BASE}/submission_churn.csv")
print(f"   예측값 범위: {test_preds.min():.4f} ~ {test_preds.max():.4f}")
print(f"   예측 이탈율: {test_preds.mean():.3%}")

IndentationError: unexpected indent (1522171890.py, line 323)

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────
# 0. 데이터 로드 (경로 유지)
# ──────────────────────────────────────────────
BASE = '/Users/rim/Desktop/workspace/project_1'

train_cust   = pd.read_csv(f'{BASE}/train/train_customer_info.csv')
train_trans  = pd.read_csv(f'{BASE}/train/train_transaction_history.csv')
train_fin    = pd.read_csv(f'{BASE}/train/train_finance_profile.csv')
train_target = pd.read_csv(f'{BASE}/train/train_targets.csv')

test_cust    = pd.read_csv(f'{BASE}/test/test_customer_info.csv')
test_trans   = pd.read_csv(f'{BASE}/test/test_transaction_history.csv')
test_fin     = pd.read_csv(f'{BASE}/test/test_finance_profile.csv')

print("✅ 데이터 로드 완료")

# ──────────────────────────────────────────────
# 1. 피처 엔지니어링 (기존 로직 유지 + 타겟 포함 처리)
# ──────────────────────────────────────────────
def build_features(cust_df, trans_df, fin_df, target_df=None):
    df = pd.merge(cust_df, fin_df, on='customer_id', how='left')
    if target_df is not None:
        df = pd.merge(df, target_df, on='customer_id', how='left')

    # 날짜 처리
    df['join_date'] = pd.to_datetime(df['join_date'])
    trans_df = trans_df.copy()
    trans_df['trans_date'] = pd.to_datetime(trans_df['trans_date'])
    trans_df['month'] = trans_df['trans_date'].dt.month
    ref_date = trans_df['trans_date'].max()
    df['days_since_joined'] = (ref_date - df['join_date']).dt.days

    # 범주형 인코딩
    cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # 파생 변수
    df['net_asset'] = df['total_deposit_balance'] - df['total_loan_balance']
    df['debt_to_deposit'] = df['total_loan_balance'] / (df['total_deposit_balance'] + 1)
    
    # 재정 위기 지표
    distress_vars = ['fin_overdue_days', 'card_cash_service_amt', 'card_loan_amt', 'total_loan_balance']
    distress_z = df[distress_vars].copy()
    for v in distress_vars:
        mu, sd = distress_z[v].mean(), distress_z[v].std() + 1e-9
        distress_z[v] = (distress_z[v] - mu) / sd
    df['fin_distress_v2'] = distress_z.sum(axis=1)

    # 거래 집계
    agg = trans_df.groupby('customer_id').agg(
        trans_count=('trans_id', 'count'),
        amt_sum=('trans_amount', 'sum'),
        amt_mean=('trans_amount', 'mean'),
        amt_std=('trans_amount', 'std'),
        recency=('trans_date', lambda x: (ref_date - x.max()).days),
    ).reset_index()
    df = pd.merge(df, agg, on='customer_id', how='left')
    df.fillna(0, inplace=True)

    # 월별 추세 (LTV 핵심)
    monthly_amt = trans_df.groupby(['customer_id','month'])['trans_amount'].sum().unstack(fill_value=0)
    monthly_amt.columns = [f'amt_m{int(c)}' for c in monthly_amt.columns]
    df = df.merge(monthly_amt.reset_index(), on='customer_id', how='left').fillna(0)

    # 타겟 인코딩 (Churn 용)
    if target_df is not None and 'target_churn' in df.columns:
        for col in ['region_code', 'prefer_category', 'income_group']:
            kf = KFold(n_splits=5, shuffle=True, random_state=42)
            df[f'{col}_churn_rate'] = 0.0
            for tr_idx, val_idx in kf.split(df):
                mapping = df.iloc[tr_idx].groupby(col)['target_churn'].mean()
                df.loc[df.index[val_idx], f'{col}_churn_rate'] = df.loc[df.index[val_idx], col].map(mapping)
            df[f'{col}_churn_rate'].fillna(df['target_churn'].mean(), inplace=True)

    return df

# 피처 리스트 (기존 유지)
FEATURES = ['total_deposit_balance', 'card_loan_amt', 'credit_score', 'fin_distress_v2', 
            'net_asset', 'fin_asset_trend_score', 'trans_count', 'amt_sum', 'amt_mean', 
            'recency', 'amt_m12', 'amt_m11', 'age', 'days_since_joined']
TE_FEATURES = ['region_code_churn_rate', 'prefer_category_churn_rate', 'income_group_churn_rate']

# ──────────────────────────────────────────────
# 2. 데이터 준비
# ──────────────────────────────────────────────
train_df = build_features(train_cust, train_trans, train_fin, train_target)
test_df  = build_features(test_cust, test_trans, test_fin, target_df=None)

# 타겟 인코딩 테스트 데이터 적용
for col in ['region_code', 'prefer_category', 'income_group']:
    mapping = train_df.groupby(col)['target_churn'].mean()
    test_df[f'{col}_churn_rate'] = test_df[col].map(mapping).fillna(train_df['target_churn'].mean())

X = train_df[FEATURES + TE_FEATURES]
y_churn = train_df['target_churn']
y_ltv = train_df['target_ltv']
X_test = test_df[FEATURES + TE_FEATURES]

# ──────────────────────────────────────────────
# 3. Churn 모델 학습 (기존 베스트 파라미터)
# ──────────────────────────────────────────────
print("\n🚀 Churn 모델 학습 시작...")
churn_params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'random_state': 42,
    'learning_rate': 0.014369, 'num_leaves': 31, 'n_estimators': 3000
}
churn_model = lgb.LGBMClassifier(**churn_params)
churn_model.fit(X, y_churn)
test_churn_preds = churn_model.predict_proba(X_test)[:, 1]

# ──────────────────────────────────────────────
# 4. LTV 모델 학습 (2-Stage 반영)
# ──────────────────────────────────────────────
print("\n🚀 LTV 2-Stage 모델 학습 시작...")

# [Step 1] VIP 분류 모델 (상위 20%)
vip_threshold = y_ltv.quantile(0.80)
y_vip = (y_ltv > vip_threshold).astype(int)

clf_vip = lgb.LGBMClassifier(objective='binary', random_state=42)
clf_vip.fit(X, y_vip)
vip_prob = clf_vip.predict_proba(X_test)[:, 1]

# [Step 2] VIP vs Normal 개별 회귀 모델 (로그 변환 적용)
# target_ltv에 log1p 적용: y' = ln(1 + target_ltv)
y_ltv_log = np.log1p(y_ltv)

reg_vip = lgb.LGBMRegressor(objective='regression', n_estimators=2000, num_leaves=127, random_state=42)
reg_normal = lgb.LGBMRegressor(objective='regression', n_estimators=2000, random_state=42)

reg_vip.fit(X[y_vip == 1], y_ltv_log[y_vip == 1])
reg_normal.fit(X[y_vip == 0], y_ltv_log[y_vip == 0])

# 예측 및 복원: expm1(y') = exp(y') - 1
pred_vip = np.expm1(reg_vip.predict(X_test))
pred_normal = np.expm1(reg_normal.predict(X_test))

# VIP 확률에 따라 가중치 결합 또는 threshold 적용 (여기서는 0.5 기준)
test_ltv_preds = np.where(vip_prob > 0.5, pred_vip, pred_normal)

# ──────────────────────────────────────────────
# 5. 최종 제출 파일 생성 (통합)
# ──────────────────────────────────────────────
submission = pd.DataFrame({
    'customer_id':  test_df['customer_id'],
    'target_churn': test_churn_preds,
    'target_ltv':   test_ltv_preds
})

submission.to_csv(f'{BASE}/submission_final.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ 최종 제출 파일 저장 완료: {BASE}/submission_final.csv")
print(f"📊 LTV 예측값 요약:")
print(submission['target_ltv'].describe())

✅ 데이터 로드 완료

🚀 Churn 모델 학습 시작...

🚀 LTV 2-Stage 모델 학습 시작...

✅ 최종 제출 파일 저장 완료: /Users/rim/Desktop/workspace/project_1/submission_final.csv
📊 LTV 예측값 요약:
count    4.000000e+04
mean     4.214487e+05
std      1.875075e+05
min      7.439980e+03
25%      2.972707e+05
50%      4.116158e+05
75%      5.266245e+05
max      3.415054e+06
Name: target_ltv, dtype: float64


In [2]:
from sklearn.metrics import mean_squared_error

# 5-Fold 교차 검증 준비
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ltv_rmse_list = []

print("\n🚀 LTV 모델 교차 검증 및 RMSE 측정 중...")

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_ltv.iloc[tr_idx], y_ltv.iloc[val_idx]
    
    # 모델 학습 (로그 변환 적용)
    model = lgb.LGBMRegressor(n_estimators=1000, random_state=42)
    model.fit(X_tr, np.log1p(y_tr))
    
    # 예측 및 복원 (expm1)
    preds = np.expm1(model.predict(X_val))
    
    # RMSE 계산
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    ltv_rmse_list.append(rmse)
    print(f" Fold {fold+1} RMSE: {rmse:,.0f}원")

print(f"✅ 평균 CV RMSE: {np.mean(ltv_rmse_list):,.0f}원")


🚀 LTV 모델 교차 검증 및 RMSE 측정 중...
 Fold 1 RMSE: 1,527,175원
 Fold 2 RMSE: 1,478,234원
 Fold 3 RMSE: 1,461,702원
 Fold 4 RMSE: 1,503,414원
 Fold 5 RMSE: 1,511,337원
✅ 평균 CV RMSE: 1,496,372원


In [3]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold, KFold

# ──────────────────────────────────────────────
# 1. Churn(이탈) 모델 성능 평가 (ROC-AUC)
# ──────────────────────────────────────────────
print("📊 [이탈 예측: Churn] 성능 평가 시작...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
churn_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_churn)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_churn.iloc[tr_idx], y_churn.iloc[val_idx]
    
    model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.01, random_state=42, verbosity=-1)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    
    preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    churn_aucs.append(auc)
    print(f"  - Fold {fold+1} AUC: {auc:.4f}")

print(f"✅ [Churn 결과] 평균 AUC: {np.mean(churn_aucs):.4f} (1에 가까울수록 우수)")


# ──────────────────────────────────────────────
# 2. LTV 모델 성능 평가 (RMSE, R2 Score)
# ──────────────────────────────────────────────
print("\n📊 [가치 예측: LTV] 성능 평가 시작...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ltv_rmses = []
ltv_r2s = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_ltv.iloc[tr_idx], y_ltv.iloc[val_idx]
    
    # 로그 변환 학습
    model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.01, random_state=42, verbosity=-1)
    model.fit(X_tr, np.log1p(y_tr))
    
    # 예측 및 지수 복원
    log_preds = model.predict(X_val)
    preds = np.expm1(log_preds)
    
    # 지표 계산
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    
    ltv_rmses.append(rmse)
    ltv_r2s.append(r2)
    print(f"  - Fold {fold+1} RMSE: {rmse:,.0f}원 | R2: {r2:.4f}")

print(f"\n✅ [LTV 결과] 최종 요약")
print(f"  - 평균 RMSE: {np.mean(ltv_rmses):,.0f}원 (0에 가까울수록 우수)")
print(f"  - 평균 R2 Score: {np.mean(ltv_r2s):.4f} (1에 가까울수록 우수)")

📊 [이탈 예측: Churn] 성능 평가 시작...
  - Fold 1 AUC: 0.7985
  - Fold 2 AUC: 0.7826
  - Fold 3 AUC: 0.7959
  - Fold 4 AUC: 0.8019
  - Fold 5 AUC: 0.7918
✅ [Churn 결과] 평균 AUC: 0.7941 (1에 가까울수록 우수)

📊 [가치 예측: LTV] 성능 평가 시작...
  - Fold 1 RMSE: 1,529,823원 | R2: -0.2134
  - Fold 2 RMSE: 1,473,660원 | R2: -0.2114
  - Fold 3 RMSE: 1,458,105원 | R2: -0.2007
  - Fold 4 RMSE: 1,497,769원 | R2: -0.2014
  - Fold 5 RMSE: 1,507,751원 | R2: -0.2043

✅ [LTV 결과] 최종 요약
  - 평균 RMSE: 1,493,422원 (0에 가까울수록 우수)
  - 평균 R2 Score: -0.2062 (1에 가까울수록 우수)


In [4]:
"""
==============================================================
  최종 통합 제출 모델 (submission_final.py)
  Churn + LTV 분리 피처 + 로그 변환 + 5-Fold OOF
==============================================================
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

BASE = '/Users/rim/Desktop/workspace/project_1'

# ──────────────────────────────────────────────
# 0. 데이터 로드
# ──────────────────────────────────────────────
train_cust   = pd.read_csv(f'{BASE}/train/train_customer_info.csv')
train_trans  = pd.read_csv(f'{BASE}/train/train_transaction_history.csv')
train_fin    = pd.read_csv(f'{BASE}/train/train_finance_profile.csv')
train_target = pd.read_csv(f'{BASE}/train/train_targets.csv')

test_cust    = pd.read_csv(f'{BASE}/test/test_customer_info.csv')
test_trans   = pd.read_csv(f'{BASE}/test/test_transaction_history.csv')
test_fin     = pd.read_csv(f'{BASE}/test/test_finance_profile.csv')

print("✅ 데이터 로드 완료")


# ──────────────────────────────────────────────
# 1. 공통 피처 엔지니어링
# ──────────────────────────────────────────────
def build_features(cust_df, trans_df, fin_df, target_df=None):
    df = pd.merge(cust_df, fin_df, on='customer_id', how='left')
    if target_df is not None:
        df = pd.merge(df, target_df, on='customer_id', how='left')

    # 날짜
    df['join_date']   = pd.to_datetime(df['join_date'])
    trans_df = trans_df.copy()
    trans_df['trans_date'] = pd.to_datetime(trans_df['trans_date'])
    trans_df['month']      = trans_df['trans_date'].dt.month
    ref_date = trans_df['trans_date'].max()
    df['days_since_joined'] = (ref_date - df['join_date']).dt.days

    # 범주형 인코딩
    for col in ['gender', 'region_code', 'prefer_category', 'income_group']:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # ── 재정 상태 ──────────────────────────────
    df['net_asset']          = df['total_deposit_balance'] - df['total_loan_balance']
    df['debt_to_deposit']    = df['total_loan_balance'] / (df['total_deposit_balance'] + 1)
    df['cash_service_ratio'] = df['card_cash_service_amt'] / (df['total_deposit_balance'] + 1)

    # 재정 위기 복합 지표
    dv = ['fin_overdue_days','card_cash_service_amt','card_loan_amt','total_loan_balance']
    dz = df[dv].copy()
    for v in dv:
        dz[v] = (dz[v] - dz[v].mean()) / (dz[v].std() + 1e-9)
    df['fin_distress_v2'] = dz.sum(axis=1)

    # ── 거래 집계 ──────────────────────────────
    agg = trans_df.groupby('customer_id').agg(
        trans_count      = ('trans_id',      'count'),
        amt_sum          = ('trans_amount',   'sum'),
        amt_mean         = ('trans_amount',   'mean'),
        amt_max          = ('trans_amount',   'max'),
        amt_std          = ('trans_amount',   'std'),
        installment_ratio= ('is_installment', 'mean'),
        online_ratio     = ('biz_type',       lambda x: (x == 'Online').mean()),
        recency          = ('trans_date',     lambda x: (ref_date - x.max()).days),
    ).reset_index()
    agg['amt_std']          = agg['amt_std'].fillna(0)
    agg['spending_per_trans']= agg['amt_sum'] / (agg['trans_count'] + 1)
    agg['spending_cv']       = agg['amt_std'] / (agg['amt_mean'] + 1)  # 변동계수

    df = pd.merge(df, agg, on='customer_id', how='left')

    # ── 월별 Wide 피처 ─────────────────────────
    monthly_amt = trans_df.groupby(['customer_id','month'])['trans_amount'] \
        .sum().unstack(fill_value=0)
    monthly_amt.columns = [f'amt_m{int(c)}' for c in monthly_amt.columns]

    monthly_cnt = trans_df.groupby(['customer_id','month'])['trans_id'] \
        .count().unstack(fill_value=0)
    monthly_cnt.columns = [f'cnt_m{int(c)}' for c in monthly_cnt.columns]

    df = df.merge(monthly_amt.reset_index(), on='customer_id', how='left')
    df = df.merge(monthly_cnt.reset_index(), on='customer_id', how='left')

    month_cols = [c for c in df.columns if c.startswith('amt_m') or c.startswith('cnt_m')]
    df[month_cols] = df[month_cols].fillna(0)

    # 후반(10-12월) vs 전반(7-9월) 소비 추세
    h1 = [c for c in df.columns if c in ['amt_m7','amt_m8','amt_m9']]
    h2 = [c for c in df.columns if c in ['amt_m10','amt_m11','amt_m12']]
    if h1 and h2:
        df['amt_first_half']    = df[h1].sum(axis=1)
        df['amt_second_half']   = df[h2].sum(axis=1)
        df['half_growth_ratio'] = df['amt_second_half'] / (df['amt_first_half'] + 1)

    # 활동 달 수
    amt_m_cols = [c for c in df.columns if c.startswith('amt_m')]
    df['active_months'] = (df[amt_m_cols] > 0).sum(axis=1)

    # 마지막 달 거래 없는 고객 (이탈 선행 신호)
    last_m = trans_df['month'].max()
    last_buyers = trans_df[trans_df['month'] == last_m]['customer_id'].unique()
    df['no_last_month_trans'] = (~df['customer_id'].isin(last_buyers)).astype(int)

    # 최근 소비 감소 플래그
    if 'amt_m12' in df.columns and 'amt_m11' in df.columns:
        df['recent_spending_drop'] = (df['amt_m12'] < df['amt_m11']).astype(int)

    # spending trend ratio
    max_month  = trans_df['month'].max()
    recent_sum = trans_df[trans_df['month'] == max_month] \
        .groupby('customer_id')['trans_amount'].sum()
    older_mean = trans_df[trans_df['month'] < max_month] \
        .groupby('customer_id')['trans_amount'].mean()
    df['spending_trend_ratio'] = (
        df['customer_id'].map(recent_sum).fillna(0) /
        (df['customer_id'].map(older_mean).fillna(1) + 1)
    )

    # ── 피어 그룹 순위 ─────────────────────────
    df['credit_rank_in_income']  = df.groupby('income_group')['credit_score'].rank(pct=True)
    df['deposit_rank_in_region'] = df.groupby('region_code')['total_deposit_balance'].rank(pct=True)
    df['age_group']   = pd.cut(df['age'], bins=[0,30,40,50,100], labels=[0,1,2,3]).astype(float)
    df['loan_rank_in_age'] = df.groupby('age_group')['card_loan_amt'].rank(pct=True)
    df['spend_rank_in_region'] = df.groupby('region_code')['amt_sum'].rank(pct=True)

    # ── LTV 특화 파생 ──────────────────────────
    df['spend_to_deposit']  = df['amt_sum'] / (df['total_deposit_balance'] + 1)
    df['monthly_avg_spend'] = df['amt_sum'] / (df['active_months'] + 1)
    df['loyalty_index']     = df['trans_count'] / (df['days_since_joined'] / 30 + 1)
    df['high_value_ratio']  = df['amt_max'] / (df['amt_mean'] + 1)

    df.fillna(0, inplace=True)
    return df


print("🔧 피처 엔지니어링 실행 중...")
train_df = build_features(train_cust, train_trans, train_fin, train_target)
test_df  = build_features(test_cust,  test_trans,  test_fin,  target_df=None)
print(f"✅ train: {train_df.shape} / test: {test_df.shape}")


# ──────────────────────────────────────────────
# 2. 피처 목록 — Churn과 LTV를 분리
# ──────────────────────────────────────────────

# [Churn 피처] 이탈 신호에 집중 — 재정 위기, 거래 감소, 비활성화
CHURN_FEATURES = [
    # 재정 위기 (이탈 직접 원인)
    'fin_distress_v2', 'fin_overdue_days', 'card_loan_amt',
    'fin_asset_trend_score', 'debt_to_deposit', 'cash_service_ratio',
    # 자산 (이탈 방어력)
    'total_deposit_balance', 'net_asset', 'credit_score', 'num_active_cards',
    # 거래 감소 신호
    'recency', 'no_last_month_trans', 'recent_spending_drop',
    'spending_trend_ratio', 'trans_count',
    'amt_m10','amt_m11','amt_m12',
    'cnt_m10','cnt_m11','cnt_m12',
    # 피어 그룹
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',
    # 고객 속성
    'age', 'is_married', 'days_since_joined',
    'gender', 'region_code', 'prefer_category', 'income_group',
]

# [LTV 피처] 소비 여력과 소비 패턴에 집중 — 재정 위기 지표 제외
LTV_FEATURES = [
    # 소비 여력
    'total_deposit_balance', 'net_asset', 'credit_score',
    'num_active_cards', 'fin_asset_trend_score',
    # 실제 소비 패턴 (LTV의 핵심)
    'amt_sum', 'amt_mean', 'amt_max', 'amt_std',
    'spending_per_trans', 'spending_cv', 'installment_ratio',
    'online_ratio', 'trans_count', 'active_months',
    # 월별 소비 전체 (7~12월)
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    'amt_first_half','amt_second_half','half_growth_ratio',
    # LTV 특화 파생
    'spend_to_deposit','monthly_avg_spend','loyalty_index','high_value_ratio',
    # 피어 그룹 (소비 기준)
    'deposit_rank_in_region','spend_rank_in_region','credit_rank_in_income',
    # 고객 속성
    'age','is_married','days_since_joined',
    'gender','region_code','prefer_category','income_group',
]

# 실제 존재하는 컬럼만 추림
CHURN_FEATURES = [f for f in CHURN_FEATURES if f in train_df.columns]
LTV_FEATURES   = [f for f in LTV_FEATURES   if f in train_df.columns]

# test에 없는 컬럼 0으로 패딩
for f in CHURN_FEATURES + LTV_FEATURES:
    if f not in test_df.columns:
        test_df[f] = 0

print(f"\n📋 Churn 피처: {len(CHURN_FEATURES)}개 / LTV 피처: {len(LTV_FEATURES)}개")


# ──────────────────────────────────────────────
# 3. 타겟 인코딩 (OOF, Churn용)
# ──────────────────────────────────────────────
def add_target_encoding(train_df, test_df, cat_cols, target_col, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for col in cat_cols:
        feat = f'{col}_churn_rate'
        train_df[feat] = 0.0
        for tr_idx, val_idx in kf.split(train_df):
            mapping = train_df.iloc[tr_idx].groupby(col)[target_col].mean()
            train_df.loc[train_df.index[val_idx], feat] = \
                train_df.loc[train_df.index[val_idx], col].map(mapping)
        train_df[feat].fillna(train_df[target_col].mean(), inplace=True)
        # test는 train 전체 통계로
        test_df[feat] = test_df[col].map(
            train_df.groupby(col)[target_col].mean()
        ).fillna(train_df[target_col].mean())
        CHURN_FEATURES.append(feat)

te_cols = ['region_code', 'prefer_category', 'income_group']
add_target_encoding(train_df, test_df, te_cols, 'target_churn')

X_churn      = train_df[CHURN_FEATURES]
X_ltv        = train_df[LTV_FEATURES]
y_churn      = train_df['target_churn']
y_ltv        = train_df['target_ltv']
y_ltv_log    = np.log1p(y_ltv)

X_test_churn = test_df[CHURN_FEATURES]
X_test_ltv   = test_df[LTV_FEATURES]


# ──────────────────────────────────────────────
# 4. Churn 모델 (5-Fold OOF)
# ──────────────────────────────────────────────
CHURN_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'random_state': 42, 'n_estimators': 5000,
    'learning_rate': 0.014369, 'num_leaves': 31,
    'min_child_samples': 22, 'feature_fraction': 0.7661,
    'bagging_fraction': 0.7553, 'bagging_freq': 6,
    'reg_alpha': 0.0172, 'reg_lambda': 0.0525,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_churn   = np.zeros(len(X_churn))
test_churn_preds = []

print("\n🚀 Churn 모델 학습 (5-Fold OOF)...")
for fold, (tr, val) in enumerate(skf.split(X_churn, y_churn)):
    m = lgb.LGBMClassifier(**CHURN_PARAMS)
    m.fit(X_churn.iloc[tr], y_churn.iloc[tr],
          eval_set=[(X_churn.iloc[val], y_churn.iloc[val])],
          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
    oof_churn[val] = m.predict_proba(X_churn.iloc[val])[:, 1]
    test_churn_preds.append(m.predict_proba(X_test_churn)[:, 1])
    print(f"  Fold {fold+1}: AUC = {roc_auc_score(y_churn.iloc[val], oof_churn[val]):.4f}")

cv_churn_auc = roc_auc_score(y_churn, oof_churn)
test_churn_final = np.mean(test_churn_preds, axis=0)
print(f"\n  ✅ Churn CV AUC: {cv_churn_auc:.4f}")


# ──────────────────────────────────────────────
# 5. LTV 모델 (5-Fold OOF, 로그 변환)
#
#  핵심 변경:
#  - Churn 피처가 아닌 LTV 전용 피처 사용
#  - log1p 변환 후 학습, expm1로 복원
#  - 단일 모델 (VIP 분리 없이) — 분리 시 VIP 데이터가 너무 적어
#    모델이 불안정해지는 문제 방지
# ──────────────────────────────────────────────
LTV_PARAMS = {
    'objective':     'regression',  # log 변환 후 일반 회귀
    'metric':        'rmse',
    'verbosity':     -1,
    'random_state':  42,
    'n_estimators':  5000,
    'learning_rate': 0.01,
    'num_leaves':    63,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_ltv_log   = np.zeros(len(X_ltv))
test_ltv_preds = []

print("\n🚀 LTV 모델 학습 (5-Fold OOF, log scale)...")
for fold, (tr, val) in enumerate(kf.split(X_ltv)):
    m = lgb.LGBMRegressor(**LTV_PARAMS)
    m.fit(X_ltv.iloc[tr], y_ltv_log.iloc[tr],
          eval_set=[(X_ltv.iloc[val], y_ltv_log.iloc[val])],
          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
    oof_ltv_log[val] = m.predict(X_ltv.iloc[val])
    test_ltv_preds.append(m.predict(X_test_ltv))

    fold_rmse = np.sqrt(mean_squared_error(
        y_ltv.iloc[val], np.expm1(oof_ltv_log[val])))
    print(f"  Fold {fold+1}: RMSE = {fold_rmse:,.0f}원")

# 복원 및 음수 방지
oof_ltv_final  = np.maximum(np.expm1(oof_ltv_log), 0)
test_ltv_final = np.maximum(np.expm1(np.mean(test_ltv_preds, axis=0)), 0)

cv_ltv_rmse  = np.sqrt(mean_squared_error(y_ltv, oof_ltv_final))
ltv_score    = 1 / (1 + np.log(cv_ltv_rmse))  # 대회 평가 공식

print(f"\n  ✅ LTV CV RMSE: {cv_ltv_rmse:,.0f}원")
print(f"  ✅ LTV 대회 점수: {ltv_score:.6f}")


# ──────────────────────────────────────────────
# 6. 예측 분포 진단 (제출 전 필수 확인)
# ──────────────────────────────────────────────
print("\n📊 예측 분포 진단")
print(f"  [Churn] 범위: {test_churn_final.min():.4f} ~ {test_churn_final.max():.4f}")
print(f"  [Churn] 평균: {test_churn_final.mean():.4f} (train 이탈율: {y_churn.mean():.4f})")
print(f"  [LTV] 범위: {test_ltv_final.min():,.0f} ~ {test_ltv_final.max():,.0f}원")
print(f"  [LTV] 평균: {test_ltv_final.mean():,.0f}원 (train 평균: {y_ltv.mean():,.0f}원)")
print(f"  [LTV] 중앙값: {np.median(test_ltv_final):,.0f}원 (train 중앙값: {y_ltv.median():,.0f}원)")

# 분포 이상 경보
if abs(test_churn_final.mean() - y_churn.mean()) > 0.02:
    print("  ⚠️ Churn 예측 평균이 train과 많이 다릅니다. 피처 누수 확인 필요")
if test_ltv_final.max() < y_ltv.quantile(0.5):
    print("  ⚠️ LTV 최대 예측값이 train 중앙값보다 낮습니다. 로그 복원 확인 필요")


# ──────────────────────────────────────────────
# 7. 최종 통합 제출 파일 생성
# ──────────────────────────────────────────────
submission = pd.DataFrame({
    'customer_id':  test_df['customer_id'],
    'target_churn': test_churn_final,
    'target_ltv':   test_ltv_final,
})

# 음수 방지 최종 처리
submission['target_churn'] = submission['target_churn'].clip(0, 1)
submission['target_ltv']   = submission['target_ltv'].clip(lower=0)

out_path = f'{BASE}/submission_final.csv'
submission.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n{'='*55}")
print(f"  🏆 Churn CV AUC : {cv_churn_auc:.4f}")
print(f"  🏆 LTV CV RMSE  : {cv_ltv_rmse:,.0f}원")
print(f"  🏆 LTV 대회점수 : {ltv_score:.6f}")
print(f"  ✅ 제출 파일 저장: {out_path}")
print(f"  ✅ 총 {len(submission):,}명 예측 완료")
print(f"{'='*55}")

✅ 데이터 로드 완료
🔧 피처 엔지니어링 실행 중...
✅ train: (60000, 61) / test: (40000, 59)

📋 Churn 피처: 31개 / LTV 피처: 44개

🚀 Churn 모델 학습 (5-Fold OOF)...
  Fold 1: AUC = 0.8021
  Fold 2: AUC = 0.7846
  Fold 3: AUC = 0.8013
  Fold 4: AUC = 0.8058
  Fold 5: AUC = 0.7986

  ✅ Churn CV AUC: 0.7961

🚀 LTV 모델 학습 (5-Fold OOF, log scale)...
  Fold 1: RMSE = 1,540,692원
  Fold 2: RMSE = 1,485,142원
  Fold 3: RMSE = 1,468,880원
  Fold 4: RMSE = 1,511,507원
  Fold 5: RMSE = 1,518,250원

  ✅ LTV CV RMSE: 1,505,106원
  ✅ LTV 대회 점수: 0.065684

📊 예측 분포 진단
  [Churn] 범위: 0.0258 ~ 0.9010
  [Churn] 평균: 0.0984 (train 이탈율: 0.0988)
  [LTV] 범위: 178,364 ~ 781,227원
  [LTV] 평균: 584,918원 (train 평균: 1,239,290원)
  [LTV] 중앙값: 631,770원 (train 중앙값: 807,154원)
  ⚠️ LTV 최대 예측값이 train 중앙값보다 낮습니다. 로그 복원 확인 필요

  🏆 Churn CV AUC : 0.7961
  🏆 LTV CV RMSE  : 1,505,106원
  🏆 LTV 대회점수 : 0.065684
  ✅ 제출 파일 저장: /Users/rim/Desktop/workspace/project_1/submission_final.csv
  ✅ 총 40,000명 예측 완료
